In [ ]:
'''
Here we tried to use Moran model to reanalyze our results got from our previous WF model. During my 3rd committee meeting, Alex
pointed out that the stair-like fitness dynamics we found in facultative sex is an artifact of WF model, as all the germ mutations
get exposed at the same time which greatly reduced the population fitness. Then selection will only work on the individuals with
very low fitness (as they just had sex). 

Here to investigate whether this artifact will matter our results, we tried to use Moran model to re-study this question. This 
new Moran model is different from Joe's previous one. In the new Moran model, the population will adpot a selection-reproduction
-mutation processes in each step (generally within a population with size N, it will take N steps to complete one generation). 

(1) The selection process is same as our WF model, which is achieved by selection with replacement based on their relative fitness.
In each selection, we only select one parent.

(2) For the reproduction process, we introduced another paramter, sex_prob. We use a binomial trial with sex_prob as occurrence 
probability to determine whether sex can occur. If sex cannot occur, then we will let the only parent selected to conduct asexual
reproduction (by either mitosis or amitosis). If sex occur, we will pick another parent randomly based on its relative fitness 
and let it reproduce sexually with the first-selected parent. No matter whether having sex, only one offspring generated will be 
maintained, and replace another randomly selected individual ("dead one") within the population. As the maximum frequency for 
Tetrahymena to have sex is around every 100 generations, the sex_prob is set to be 0.01.

(3) For the mutation process, we will only mutate the maintained offspring generated in reproduction process.

At the same time, we monitored the age of individuals within the population. Once the selected parents participate in reproduction,
their age will be added by 1. For the offspring generated, if it is generated by sex, then the age of offspring will be set to 0.
Otherwise, the age of offspring will be the age of its parents plus 1. Here the age is monitored by the number of steps, dividing
population size N will get the age shown by generation. But in analyses, we mainly foucused on fitness dynamics but didn't focus 
on the age data.

In this code, this is assuming having both beneficial and deleterious mutations, and the asexual reproduction is conducted by 
amitosis.


'''

In [ ]:
from __future__ import division
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import time
import pickle
import multiprocessing as mp



class Population(object):

    
    def __init__(self,nInds,nLoci,ploidy=45, genomic_mu=0.1,beneficial_selcoef=0.1, deleterious_selcoef = 0.1, \
                 beneficial_prop = 0.5, sex_prob =0.01):

        self.nInds = nInds
        self.nLoci = nLoci
  
        self.soma = np.zeros((nInds,nLoci),dtype='int')
        self.soma_beneficial = np.zeros((nInds,nLoci),dtype='int') # store the beneficial mutations
        self.soma_deleterious = np.zeros((nInds,nLoci),dtype='int') # store the deleterious mutations


        self.germ = np.zeros((nInds,nLoci),dtype='int')
        self.germ_beneficial = np.zeros((nInds,nLoci),dtype='int')
        self.germ_deleterious = np.zeros((nInds,nLoci),dtype='int')  


        self.ploidy = ploidy
        self.mu = genomic_mu/(nLoci*ploidy)
        self.beneficial_selcoef = beneficial_selcoef
        self.deleterious_selcoef = deleterious_selcoef
        
        self.beneficial_prop = beneficial_prop

        self.pop_age = np.zeros(self.nInds, dtype = 'int')
      
        self.sex_prob = sex_prob
        self.current_step = 0




    def fitness(self):
        """return a numpy array containing the fitness for each individual in each population"""
        bene_ws = (self.soma_beneficial.astype('float')/self.ploidy)*self.beneficial_selcoef
        dele_ws = (self.soma_deleterious.astype('float')/self.ploidy)*self.deleterious_selcoef

        each_locus = 1 + bene_ws - dele_ws
        fitnesses = np.prod(each_locus, axis =1)

        return fitnesses


            
    def relative_fitness(self):
        """return a numpy array containing each individual's relative fitness"""
        w = self.fitness()
        total_w = np.sum(w)
        totalw = np.expand_dims(total_w,axis=0)
        relfit = w/totalw
        return relfit
                

        

    # Moran model
    
    def pick_parent(self):
        """Randomly choose one parent per population to produce offspring for the next
        time step. Each individual's probability of being chosen is weighted by its 
        relative fitness.
        Moran model method."""
        relfit = self.relative_fitness()
        csrelfit = np.cumsum(relfit,axis=0)
        randvals = np.random.random(1)
        parents = np.searchsorted(csrelfit,randvals)[0]
        return parents
    


    def moran_step(self):

        parent1 = self.pick_parent()
        self.pop_age[parent1] += 1

        self.sex_trial = np.random.binomial(1, self.sex_prob)


        if self.sex_trial == 1:

            parent2 = self.pick_parent()
            self.pop_age[parent2]+=1

            offspring = self.have_sex(parent1, parent2)
            mut_offspring = self.mutate(offspring)


        else:
            # self.mutate(parent1)

            offspring = self.amitosis(parent1)
            mut_offspring = self.mutate(offspring)

        dead = np.random.randint(0,self.nInds)

        self.soma[dead] = mut_offspring[0]
        self.soma_beneficial[dead] = mut_offspring[1]
        self.soma_deleterious[dead] = mut_offspring[2]

        self.germ[dead] = mut_offspring[3]
        self.germ_beneficial[dead] = mut_offspring[4]
        self.germ_deleterious[dead] = mut_offspring[5]


        if self.sex_trial ==1:
            self.pop_age[dead] =0
        else:
            self.pop_age[dead] = self.pop_age[parent1].copy()
        
        self.current_step+=1 


    def amitosis(self,parent):
        """Generate amitotic offspring from all individuals selected as parents. Only one
        amitotic offspring is generated from each parent, so this method does not reflect
        the reciprocity of amitosis.
        Wright-Fisher model method."""
        
        # may consider multi hypergeometric distribution
        
        
        good = (self.ploidy-self.soma[parent])*2  # wild type alleles in each locus
        bad = self.soma[parent]*2  # mutated alleles (bene+dele) in each locus
        
        bad_bene = self.soma_beneficial[parent]*2
        bad_dele = self.soma_deleterious[parent]*2
        
        new_soma = self.ploidy - np.random.hypergeometric(good, bad, self.ploidy) # np.random.hypergeometric(good, bad, 
                                                             # \self.ploidy) is the number of wild type alleles in each locus.
            # self.ploidy - np.random.hypergeometric(good, bad, self.ploidy) is the number of mutated alleles (bene + dele) in
            # each locus

        new_soma_bene = np.zeros(self.nLoci, dtype ='int')
        new_soma_dele = np.zeros(self.nLoci, dtype ='int')
        
        new_soma_bene[new_soma!=0]= np.random.hypergeometric(bad_bene[new_soma!=0], bad_dele[new_soma!=0], \
                                                                                 new_soma[new_soma!=0])
        new_soma_bene[new_soma==0]=0
        new_soma_dele = new_soma - new_soma_bene

        new_germ = self.germ[parent].copy()
        new_germ_bene = self.germ_beneficial[parent].copy()
        new_germ_dele = self.germ_deleterious[parent].copy()

        return new_soma, new_soma_bene, new_soma_dele, new_germ, new_germ_bene, new_germ_dele



    def make_gametes(self, parent):

        gametes = 1- np.random.hypergeometric(2-self.germ[parent],self.germ[parent],1) 
        # np.random.hypergeometric(2-self.germ[rReps,parents,],self.germ[rReps,parents,],1) stores how many wild type alleles 
        # remains in each locus. 1- np.random.hypergeometric(2-self.germ[rReps,parents,],self.germ[rReps,parents,],1) stores 
        # the number of mutations in each locus.
        
        gametes_bene = np.zeros((self.nLoci),dtype='int')  # initialize beneficial and deleterious 
        gametes_dele = np.zeros((self.nLoci),dtype='int')  # in gametes
        
        gametes_bene[gametes!=0] = np.random.hypergeometric(self.germ_beneficial[parent][gametes!=0], \
                                                            self.germ_deleterious[parent][gametes!=0],\
                                                            gametes[gametes!=0])
        # For locus where gametes ==0, it means that locus has no mutations. Thus also have no beneficial and deleterious 
        # mutations.
        
        gametes_dele = gametes - gametes_bene
        
        return gametes, gametes_bene, gametes_dele



    def form_zygotes(self, gamete1_total, gamete2_total):

        gamete1 = gamete1_total[0]
        gamete1_bene = gamete1_total[1]
        gamete1_dele = gamete1_total[2]

        gamete2 = gamete2_total[0]
        gamete2_bene = gamete2_total[1]
        gamete2_dele = gamete2_total[2]

        zygotes = gamete1 + gamete2
        zy_bene = gamete1_bene + gamete2_bene
        zy_dele = gamete1_dele + gamete2_dele

        return zygotes, zy_bene, zy_dele



    def have_sex(self, parent1, parent2):

        # First need to generate gametes from MIC
        gamete1 = self.make_gametes(parent1)
        gamete2 = self.make_gametes(parent2)


        # Then two gametes get fused into a diploid zygotes
        zy = self.form_zygotes(gamete1, gamete2)

        # Then develop into new individual
        zygote = zy[0]
        zy_bene = zy[1]
        zy_dele = zy[2]

        new_germ = zygote.copy()
        new_germ_bene = zy_bene.copy()
        new_germ_dele = zy_dele.copy()

        new_soma = np.zeros(self.nLoci, dtype = 'int')
        new_soma_bene = np.zeros(self.nLoci, dtype = 'int')
        new_soma_dele = np.zeros(self.nLoci, dtype = 'int')

        new_soma[new_germ ==0] =0
        new_soma_bene[new_germ ==0]=0
        new_soma_dele[new_germ ==0]=0


        for i in range(self.nLoci):
            if new_germ[i] ==1 and new_germ_bene[i]==1:
                new_soma[i] = int(self.ploidy/2) + np.random.binomial(self.ploidy-2*int(self.ploidy/2),0.5)
                new_soma_bene[i] = new_soma[i].copy()
                new_soma_dele[i] =0       

            elif new_germ[i] ==1 and new_germ_dele[i]==1:
                new_soma[i] = int(self.ploidy/2) + np.random.binomial(self.ploidy-2*int(self.ploidy/2),0.5)
                new_soma_bene[i] = 0
                new_soma_dele[i] =new_soma[i].copy()   

            elif new_germ[i]==2: # this locus is mutation fixed.
                new_soma[i]=self.ploidy  # mutation fixed in this locus
                if new_germ_bene[i] == 2:
                    new_soma_bene[i]=self.ploidy
                elif new_germ_dele[i] ==2:
                    new_soma_dele[i] = self.ploidy
                else:
                    if new_germ_bene[i]!=1 or new_germ_dele[i]!=1:
                        print 'ERROR'
                    new_soma_bene[i] = int(self.ploidy/2) + np.random.binomial(self.ploidy-2*int(self.ploidy/2),0.5)
                    new_soma_dele[i] = self.ploidy - new_soma_bene[i]


        return new_soma, new_soma_bene, new_soma_dele, new_germ, new_germ_bene, new_germ_dele





    def mutate(self,individual):
        """Mutate individuals in 'parents' (one individual per population)
        with mutation rate mu/2. Mutation rate is halved because both offspring will
        carry the mutations of the parent.
        Moran model method"""

        ind_soma = individual[0]
        ind_soma_bene = individual[1]
        ind_soma_dele = individual[2]

        ind_germ = individual[3]
        ind_germ_bene = individual[4]
        ind_germ_dele = individual[5]   


        wt_soma = self.ploidy - ind_soma   # the still mutable alleles in each loci
        wt_germ = 2 - ind_germ
        
        soma_mutations = np.random.binomial(wt_soma,self.mu) # generate new mutations
        germ_mutations = np.random.binomial(wt_germ,self.mu) 
        
        soma_bene_mu = np.random.binomial(soma_mutations, self.beneficial_prop)  # How many of the new mutations are beneficial
        soma_dele_mu = soma_mutations - soma_bene_mu  # and deleterious
        
        germ_bene_mu = np.random.binomial(germ_mutations, self.beneficial_prop)
        germ_dele_mu = germ_mutations - germ_bene_mu
        
        ind_soma += soma_mutations   # update the soma and germ (all mutations, beneficial and deleterious mutations)
        ind_germ += germ_mutations  # how many mutations have been generated
        
        ind_soma_bene += soma_bene_mu   
        ind_germ_bene += germ_bene_mu      
        
        ind_soma_dele += soma_dele_mu   
        ind_germ_dele += germ_dele_mu  
    
        return ind_soma, ind_soma_bene, ind_soma_dele, ind_germ, ind_germ_bene, ind_germ_dele



    def get_results(self):

        W = self.fitness()

        W_mean = np.nanmean(W)
        W_var = np.var(W)

        pop_age_mean = np.nanmean(self.pop_age)
        pop_age_var = np.var(self.pop_age)

        return [W_mean, W_var, pop_age_mean, pop_age_var]


        
    def simulate(self,stepcount, strides=10):
 
        results = [self.get_results()]
        
        start = time.time()

        
        while self.current_step <= stepcount:
            self.moran_step()
            if self.current_step%strides == 0:
                results.append(self.get_results())


        colnames = ['MeanFit','VarFit', 'MeanPopAge', 'VarPopAge']

        results = pd.DataFrame(np.array(results),columns=colnames)


        end = time.time()
        print 'TOTAL TIME: ',end-start
        return results


def save_object(obj, filename):
    with open(filename, 'wb') as output:
        pickle.dump(obj, output, pickle.HIGHEST_PROTOCOL)



def get_result(data_list, data_name):

    '''Get the data from the stored data_list according to their name'''

    n = len(data_list)
    mean_data = []

    for i in range(n):
        mean_data.append(list(data_list[i][data_name]))

    m = len(list(data_list[0][data_name]))
    # print 'M', m

    each_gen_mean_data = []

    each_gen_std_data = [] # Need to check

    for j in range(m):
        each_gen_data = []
        for s in range(n):
            each_gen_data.append(mean_data[s][j])

        each_gen_mean_data.append(np.nanmean(each_gen_data))
        each_gen_std_data.append(np.nanstd(each_gen_data))  # Need check
  
    return each_gen_mean_data, each_gen_std_data




def worker(n):
    '''define a function worker to conduct a single task. Here the worker was used to run a single replicate for the
    whole simulation process. Here for I am using object-oriented programming, to let the code run, the function worker 
    is used to run the object. The function worker should have parameters otherwise it will raise error when running. But 
    the parameter n here won't be used later (here n is just to make sure that the function has parameters)'''

    np.random.seed()

    a =Population(nInds =1000,nLoci =100,ploidy=45, genomic_mu=0.1,beneficial_selcoef=0.1, deleterious_selcoef = 0.1, \
                 beneficial_prop = 0.01, sex_prob =0.01)

    results = a.simulate(stepcount =1000*1000, strides=1000)

    return results

In [ ]:
if __name__ == "__main__":  # add this line can make sure that multiprocess can be tested in Windows PC. 
                    

    start = time.time()

    pool = mp.Pool(5)   # creat a multiprocessing pool with 4 CPU (should be identical to the number of requested
                           # CPU in the .sh file, or = requested CPU-1)

    print 'CPU COUNT', mp.cpu_count()  
        
    results = pool.map_async(worker, list(range(500))) # most time consuming part, use parallel computing
                                                # Here map_async is asynchronical function, and what this function do
                                                # is to map asynchronical function to our defined function worker.
                                                # list(range(10)) is the parameters used for worker, and here it means run
                                                # worker for 10 times (i.e., the number of replicates)


    results = results.get()  # get the results
 
    fitness = [i for i in results]

 
    gen_mfit = get_result(fitness, 'MeanFit')
    gen_mfit_mean = gen_mfit[0]
    gen_mfit_std = gen_mfit[1]


    gen_varfit = get_result(fitness, 'VarFit')
    gen_varfit_mean = gen_varfit[0]
    gen_varfit_std = gen_varfit[1]


    gen_mage = get_result(fitness, 'MeanPopAge')
    gen_mage_mean = gen_mage[0]
    gen_mage_std = gen_mage[1]

    gen_varage = get_result(fitness, 'VarPopAge')
    gen_varage_mean = gen_varage[0]
    gen_varage_std = gen_varage[1]


    fitness_result = []

    for j in range(len(gen_mfit_mean)):

        fitness_result.append([gen_mfit_mean[j], gen_mfit_std[j], gen_varfit_mean[j], gen_varfit_std[j], \
            gen_mage_mean[j], gen_mage_std[j], gen_varage_mean[j], gen_varage_std[j]])


    colnames = ['PopMeanFit_Mean','PopMeanFit_Std', 'PopVarFit_Mean', 'PopVarFit_Std', \
    'PopMeanAge_Mean', 'PopMeanAge_Std', 'PopVarAge_Mean', 'PopVarAge_Std']

    fitness_result = pd.DataFrame(np.array(fitness_result),columns=colnames)


    fitness_result.to_csv('Fit_RM_E100_1KG_Bene01_Moran_181206_500Rep.csv', index =False)

    end = time.time()
    print 'TOTAL TIME:', end-start            